# Use SPSS and batch deployment with DB2 to predict customer churn with `ibm-watsonx-ai`

This notebook contains steps to deploy a sample SPSS stream and start batch scoring new data. 

Some familiarity with bash is helpful. This notebook uses Python 3.12.

You will use a data set, **Telco Customer Churn**, which details anonymous customer data from a telecommunication company. Use the details of this data set to predict customer churn. This is critical to business, as it's easier to retain existing customers than acquire new ones.

## Learning goals

The learning goals of this notebook are:

-  Loading a CSV file into Db2 on Cloud 
-  Working with the watsonx.ai Runtime instance
-  Batch deployment of an SPSS model
-  Scoring data using deployed model and a Db2 connection


## Contents

This notebook contains the following parts:

1.	[Set up the environment](#Set-up-the-environment)
2.  [Create db2 connection](#Create-a-Db2-connection)
3.	[Model upload](#Upload-model) 
4.	[Create batch deployment](#Create-batch-deployment)
5.	[Scoring](#Scoring)
6.  [Clean up](#Clean-up)
7.	[Summary and next steps](#Summary-and-next-steps)

<a id="Set-up-the-environment"></a>
## Set up the environment

Before you use the sample code in this notebook:

-  Create a <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance (a free plan is offered and information about how to create the instance can be found <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/wml-plans.html?context=wx&audience=wdp" target="_blank" rel="noopener no referrer">here</a>).

### Install dependencies
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install wget | tail -n 1
%pip install -U ibm-watsonx-ai | tail -n 1

### Connection to watsonx.ai Runtime

Authenticate the watsonx.ai Runtime service on IBM Cloud. You need to provide platform `api_key` and instance `location`.

You can use [IBM Cloud CLI](https://cloud.ibm.com/docs/cli/index.html) to retrieve platform API Key and instance location.

API Key can be generated in the following way:
```
ibmcloud login
ibmcloud iam api-key-create API_KEY_NAME
```

In result, get the value of `api_key` from the output.


Location of your watsonx.ai Runtime instance can be retrieved in the following way:
```
ibmcloud login --apikey API_KEY -a https://cloud.ibm.com
ibmcloud resource service-instance INSTANCE_NAME
```

In result, get the value of `location` from the output.

**Tip**: Your `Cloud API key` can be generated by going to the [**Users** section of the Cloud console](https://cloud.ibm.com/iam#/users). From that page, click your name, scroll down to the **API Keys** section, and click **Create an IBM Cloud API key**. Give your key a name and click **Create**, then copy the created key and paste it below. You can also get a service specific url by going to the [**Endpoint URLs** section of the watsonx.ai Runtime docs](https://cloud.ibm.com/apidocs/machine-learning).  You can check your instance location in your  <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance details.

You can also get service specific apikey by going to the [**Service IDs** section of the Cloud Console](https://cloud.ibm.com/iam/serviceids).  From that page, click **Create**, then copy the created key and paste it below.

**Action**: Enter your `url` and `api_key` in the following cell.

In [2]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key=getpass.getpass("Please enter your watsonx.ai api key (hit enter): "),
)

In [3]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials)

### Working with spaces

First, create a space that will be used for your work. If you do not have a space, you can use [Deployment Spaces Dashboard](https://dataplatform.cloud.ibm.com/ml-runtime/spaces?context=cpdaas) to create one.

- Click New Deployment Space
- Create an empty space
- Select Cloud Object Storage
- Select watsonx.ai Runtime instance and press Create
- Copy `space_id` and paste it below

**Tip**: You can also use SDK to prepare the space for your work. More information can be found [here](https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/instance-management/Space%20management.ipynb).

**Action**: Assign space ID below

In [4]:
space_id = "PASTE YOUR SPACE ID HERE"

You can use `list` method to print all existing spaces.

In [ ]:
client.spaces.list(limit=10)

To be able to interact with all resources available in watsonx.ai Runtime, you need to set **space** which you will be using.

In [5]:
client.set.default_space(space_id)

'SUCCESS'

<a id="Create-a-Db2-connection"></a>
## Create a Db2 connection
You can use commands below to create a Db2 connection and required data assets to perform batch scoring.

### Create tables in Db2 on Cloud

 - Download [inputScore.csv](https://github.com/IBM/watsonx-ai-samples/raw/master/cloud/data/customer_churn/scoreInput.csv) and [inputScore2.csv](https://github.com/IBM/watsonx-ai-samples/raw/master/cloud/data/customer_churn/scoreInput2.csv) files from the GitHub repository.
 - Click the **Open the console to get started with Db2 on Cloud** icon.
 - Select the **Load Data** and **Desktop** load type.
 - Drag and drop the previously downloaded file and click Next.
 - Set table names to `CUSTOMER` and `CUSTOMER_2` and proceed with creating.

In [6]:
import os

input_table_1 = "CUSTOMER"
input_table_2 = "CUSTOMER_2"
output_table = "OUTPUT"

try:
    schema_name = os.environ["DB_SCHEMA"]
except KeyError:
    schema_name = input("Please enter name of the schema you want to use: ")

#### Create Db2 connection asset

In [7]:
db2_connection_id = (
    input(
        "Provide connection asset ID in your space. Skip this, if you wish to type credentials by hand and hit enter: "
    )
    or None
)

if db2_connection_id is None:
    db_type = "db2"

    try:
        hostname = os.environ["DB_HOSTNAME"]
    except KeyError:
        hostname = input(
            "Please enter hostname or IP address of your database and hit enter: "
        )

    try:
        port = os.environ["DB_PORT"]
    except KeyError:
        port = input("Please enter your database port number and hit enter: ")

    try:
        database = os.environ["DB_DATABASE"]
    except KeyError:
        database = input("Please enter your database name and hit enter: ")

    try:
        username = os.environ["DB_USERNAME"]
    except KeyError:
        username = input("Please enter your username and hit enter: ")

    try:
        password = os.environ["DB_PASSWORD"]
    except KeyError:
        password = getpass.getpass("Please enter your password and hit enter: ")

    try:
        ssl = os.environ["DB_SSL"]
    except KeyError:
        ssl = getpass.getpass("Please enter your ssl certificate and hit enter: ")

    # Create connection
    db_data_source_type_id = client.connections.get_datasource_type_uid_by_name(db_type)
    details = client.connections.create(
        {
            client.connections.ConfigurationMetaNames.NAME: "Knowledge database connection",
            client.connections.ConfigurationMetaNames.DESCRIPTION: "Connection created by the sample notebook",
            client.connections.ConfigurationMetaNames.DATASOURCE_TYPE: db_data_source_type_id,
            client.connections.ConfigurationMetaNames.PROPERTIES: {
                "host": hostname,
                "port": port,
                "username": username,
                "password": password,
                "database": database,
                "ssl": True,
            },
        }
    )

    db2_connection_id = client.connections.get_id(details)

#### Create input connection data assets

In [8]:
db2_asset_meta_props = {
    client.data_assets.ConfigurationMetaNames.NAME: "INPUT_TABLE_1",
    client.data_assets.ConfigurationMetaNames.CONNECTION_ID: db2_connection_id,
    client.data_assets.ConfigurationMetaNames.DESCRIPTION: "db2 table",
    client.data_assets.ConfigurationMetaNames.DATA_CONTENT_NAME: input_table_1,
}

db2_conn_input_asset_details = client.data_assets.store(db2_asset_meta_props)
input_data_1_href = client.data_assets.get_href(db2_conn_input_asset_details)

Creating data asset...
SUCCESS


In [9]:
db2_asset_meta_props = {
    client.data_assets.ConfigurationMetaNames.NAME: "INPUT_TABLE_2",
    client.data_assets.ConfigurationMetaNames.CONNECTION_ID: db2_connection_id,
    client.data_assets.ConfigurationMetaNames.DESCRIPTION: "db2 table",
    client.data_assets.ConfigurationMetaNames.DATA_CONTENT_NAME: input_table_2,
}

db2_conn_input_asset_details = client.data_assets.store(db2_asset_meta_props)
input_data_2_href = client.data_assets.get_href(db2_conn_input_asset_details)

Creating data asset...
SUCCESS


#### Create output connection data assets

In [10]:
db2_asset_meta_props = {
    client.data_assets.ConfigurationMetaNames.NAME: "OUTPUT_TABLE",
    client.data_assets.ConfigurationMetaNames.CONNECTION_ID: db2_connection_id,
    client.data_assets.ConfigurationMetaNames.DESCRIPTION: "db2 table",
    client.data_assets.ConfigurationMetaNames.DATA_CONTENT_NAME: output_table,
}

db2_conn_output_asset_details = client.data_assets.store(db2_asset_meta_props)
output_data_href = client.data_assets.get_href(db2_conn_output_asset_details)

Creating data asset...
SUCCESS


<a id="Upload-model"></a>
## Upload model

In this section you will learn how to upload the model to the Cloud.

**Action**: Download sample SPSS model from git repository using `wget`.

In [11]:
import wget

filename = "db2-customer-satisfaction-prediction.str"
if not os.path.isfile(filename):
    filename = wget.download(
        "https://github.com/IBM/watsonx-ai-samples/raw/master/cloud/models/spss/db2_customer_satisfaction/model/db2-customer-satisfaction-prediction.str",
    )

print(filename)

db2-customer-satisfaction-prediction.str


### Update model to use provided schema

**Note:** This step is relevant only for the model specific to this notebook. In general case, you should specify your schema directly in SPSS.

In [12]:
import zipfile

new_filename = filename + ".new"

with (
    zipfile.ZipFile(filename, "r") as old_zip_file,
    zipfile.ZipFile(new_filename, "w") as new_zip_file,
):
    for item in old_zip_file.infolist():
        with old_zip_file.open(item, "r") as file:
            file_content = file.read()

        if item.filename in {"data/0001.dat", "data/0002.dat", "data/0004.dat"}:
            file_content = file_content.replace(
                b"/CJB94327/", f"/{schema_name}/".encode()
            )

        with new_zip_file.open(item, "w") as file:
            file.write(file_content)

os.rename(new_filename, filename)

Store SPSS sample model in your watsonx.ai Runtime instance.

In [13]:
sw_spec_id = client.software_specifications.get_id_by_name("spss-modeler_18.2")

model_meta_props = {
    client.repository.ModelMetaNames.NAME: "SPSS customer satisfaction model",
    client.repository.ModelMetaNames.TYPE: "spss-modeler_18.2",
    client.repository.ModelMetaNames.SOFTWARE_SPEC_ID: sw_spec_id,
}

model_details = client.repository.store_model(filename, model_meta_props)

**Note:** You can see that model is successfully stored in watsonx.ai Runtime Service.

In [14]:
client.repository.list_models()

,ID,NAME,CREATED,TYPE,SPEC_STATE,SPEC_REPLACEMENT
0,79e9d2ce-facf-4bba-bdcb-bda7dfe55863,SPSS customer satisfaction model,2026-02-23T08:01:03Z,spss-modeler_18.2,supported,


<a id="Create-batch-deployment"></a>
## Create batch deployment
Run the following cell to create batch deployment for stored model.

In [15]:
model_id = client.repository.get_model_id(model_details)

deployment = client.deployments.create(
    artifact_id=model_id,
    meta_props={
        client.deployments.ConfigurationMetaNames.NAME: "SPSS BATCH customer satisfaction",
        client.deployments.ConfigurationMetaNames.BATCH: {},
        client.deployments.ConfigurationMetaNames.HARDWARE_SPEC: {
            "name": "S",
            "num_nodes": 1,
        },
    },
)



######################################################################################

Synchronous deployment creation for id: '79e9d2ce-facf-4bba-bdcb-bda7dfe55863' started

######################################################################################


ready.


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='8ba9dcdb-08bc-4aaa-8fa0-f80dc8cdda0d'
-----------------------------------------------------------------------------------------------




<a id="Scoring"></a>
## Scoring

Create deployment job

In [16]:
job_payload_ref = {
    client.deployments.ScoringMetaNames.INPUT_DATA_REFERENCES: [
        {
            "id": "conn_db2",
            "name": "input_table_1",
            "type": "connection_asset",
            "connection": {"id": db2_connection_id},
            "location": {"schema_name": schema_name, "file_name": input_table_1},
        },
        {
            "id": "conn_db2",
            "name": "input_table_2",
            "type": "connection_asset",
            "connection": {"id": db2_connection_id},
            "location": {"schema_name": schema_name, "file_name": input_table_2},
        },
    ],
    client.deployments.ScoringMetaNames.OUTPUT_DATA_REFERENCE: {
        "id": "conn_db2",
        "name": "output_table",
        "type": "connection_asset",
        "connection": {"id": db2_connection_id},
        "location": {"schema_name": schema_name, "file_name": output_table},
    },
}

deployment_id = client.deployments.get_id(deployment)
job_details = client.deployments.create_job(deployment_id, meta_props=job_payload_ref)

Retrieve job ID

In [17]:
job_id = client.deployments.get_job_id(job_details)

##### Monitor job execution

In [18]:
import time

start_time = time.time()
while time.time() - start_time < 300:
    time.sleep(10)

    job_status = str(client.deployments.get_job_status(job_id).get("state"))

    if job_status == "completed" or "fail" in job_status:
        print(f"\nJob finished with status: {job_status}")
        print(client.deployments.get_job_details(job_id))
        break

    print(".", end="", flush=True)
else:
    print("Job hasn't completed successfully in 5 minutes.")

............
Job finished with status: completed
{'entity': {'deployment': {'id': '8ba9dcdb-08bc-4aaa-8fa0-f80dc8cdda0d'}, 'platform_job': {'job_id': 'fc8b9526-ef23-486f-8a41-d5af4ff958a1', 'run_id': 'e9c2e737-fb9c-4521-a36a-618d94653f02'}, 'scoring': {'input_data_references': [{'connection': {'id': 'ce2fbb1b-f31f-464e-8aaa-6fe550804142'}, 'id': 'conn_db2', 'location': {'file_name': 'CUSTOMER', 'schema_name': 'SPSS_NOTEBOOK'}, 'type': 'connection_asset'}, {'connection': {'id': 'ce2fbb1b-f31f-464e-8aaa-6fe550804142'}, 'id': 'conn_db2', 'location': {'file_name': 'CUSTOMER_2', 'schema_name': 'SPSS_NOTEBOOK'}, 'type': 'connection_asset'}], 'output_data_reference': {'connection': {'id': 'ce2fbb1b-f31f-464e-8aaa-6fe550804142'}, 'id': 'conn_db2', 'location': {'file_name': 'OUTPUT', 'schema_name': 'SPSS_NOTEBOOK'}, 'type': 'connection_asset'}, 'status': {'completed_at': '2026-02-23T08:03:51.062Z', 'running_at': '2026-02-23T08:02:56.603Z', 'state': 'completed'}}}, 'metadata': {'created_at': '20

#### Preview scored data

In this subsection you will load scored data.

In [19]:
from ibm_watsonx_ai.helpers.connections import DatabaseLocation, DataConnection

connection = DataConnection(
    connection_asset_id=db2_connection_id,
    location=DatabaseLocation(schema_name, output_table),
)
connection.set_client(client)
connection.read()

  Using cached pyarrow-23.0.1-cp312-cp312-macosx_12_0_arm64.whl.metadata (3.1 kB)
Using cached pyarrow-23.0.1-cp312-cp312-macosx_12_0_arm64.whl (34.2 MB)


,customerID,Churn,Predicted Churn,Probability of Churn
0,3638-WEABW,No,No,0.052631
1,5919-TMRGD,Yes,Yes,0.894228
2,9979-RGMZT,No,No,0.084381
0,4080-IIARD,No,No,0.092014
1,6575-SUVOI,No,No,0.092092
2,7495-OOKFY,Yes,Yes,0.972150
3,9102-OXKFY,No,No,0.092013
0,9237-HQITU,Yes,Yes,0.882983
1,8665-UTDHZ,Yes,No,0.174110
2,9364-YKUVW,No,No,0.085739


<a id="Clean-up"></a>
## Clean up

If you want to clean up all created assets:
- experiments
- trainings
- pipelines
- model definitions
- models
- functions
- deployments

see the steps in this sample [notebook](https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/instance-management/Machine%20Learning%20artifacts%20management.ipynb).

<a id="Summary-and-next-steps"></a>
## Summary and next steps

 You successfully completed this notebook! You learned how to use watsonx.ai Runtime for SPSS model deployment and scoring. Check out our [Online Documentation](https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/welcome-main.html?context=wx) for more samples, tutorials, documentation, how-tos, and blog posts. 

### Author

**Jan Sołtysik** Intern at watsonx.ai.

**Rafał Chrzanowski** Software Engineer at watsonx.ai

Copyright © 2020-2026 IBM. This notebook and its source code are released under the terms of the MIT License.